# 01 — Exploratory Data Analysis
**Movie & Anime Recommender System**  
Combined MovieLens (100K) + MyAnimeList dataset — distributions, genres, user behaviour.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── DESIGN.md palette ─────────────────────────────────────────────────────────
CORAL    = '#ff5530'
BLUE     = '#1456f0'
MAGENTA  = '#ea5ec1'
PURPLE   = '#a855f7'
INK      = '#0a0a0a'
STEEL    = '#5f5f5f'
HAIRLINE = '#e5e7eb'
CANVAS   = '#ffffff'
SURFACE  = '#f7f8fa'

SRC_COLORS = {'Anime': CORAL, 'Movie': BLUE}

plt.rcParams.update({
    'figure.facecolor': CANVAS,
    'axes.facecolor':   CANVAS,
    'axes.edgecolor':   HAIRLINE,
    'axes.labelcolor':  INK,
    'axes.titlepad':    12,
    'axes.titleweight': 'semibold',
    'axes.titlecolor':  INK,
    'xtick.color':      STEEL,
    'ytick.color':      STEEL,
    'text.color':       INK,
    'grid.color':       HAIRLINE,
    'grid.linewidth':   0.8,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

DATA_PATH = Path('../data/combined_ratings.csv')
print(f'Data path exists: {DATA_PATH.exists()}')

## Dataset Overview

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
overview = df.groupby('source').agg(
    ratings   = ('rating', 'count'),
    users     = ('user_id', 'nunique'),
    items     = ('item_id', 'nunique'),
    min_rating= ('rating', 'min'),
    max_rating= ('rating', 'max'),
    mean_rating=('rating', lambda x: round(x.mean(), 2)),
).reset_index()

print('\n=== Dataset Overview ===')
print(overview.to_string(index=False))
print(f'\nTotal ratings : {len(df):,}')
print(f'Total users   : {df["user_id"].nunique():,}')
print(f'Total items   : {df["item_id"].nunique():,}')

## Rating Distributions
Anime uses a 1–10 scale; MovieLens uses 1–5. Both are normalized to [0, 1] during training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (src, color) in zip(axes, SRC_COLORS.items()):
    subset = df[df['source'] == src]['rating']
    bins   = np.arange(subset.min(), subset.max() + 2) - 0.5
    ax.hist(subset, bins=bins, color=color, edgecolor=CANVAS, linewidth=0.6, alpha=0.9)
    ax.set_title(f'{src} Ratings  (n={len(subset):,})')
    ax.set_xlabel('Rating')
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.set_xticks(sorted(subset.unique()))
    ax.grid(axis='y', alpha=0.6)

fig.suptitle('Rating Distributions by Source', fontsize=14, fontweight='semibold', y=1.02)
plt.tight_layout()
plt.savefig('../model/eda_rating_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Top Genres

In [ ]:
def top_genres(source: str, n: int = 15) -> pd.Series:
    subset = df[df['source'] == source][['item_id', 'genre']].drop_duplicates('item_id')
    genres = subset['genre'].dropna().str.split(', ').explode().str.strip()
    genres = genres[genres != '']
    return genres.value_counts().head(n)

anime_genres = top_genres('Anime')
movie_genres = top_genres('Movie')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, genres, color) in zip(axes, [
    ('Anime — Top Genres (by unique titles)', anime_genres, CORAL),
    ('Movies — Top Genres (by unique titles)', movie_genres, BLUE),
]):
    bars = ax.barh(genres.index[::-1], genres.values[::-1], color=color, alpha=0.88)
    ax.set_title(title)
    ax.set_xlabel('Number of Titles')
    ax.bar_label(bars, padding=4, fontsize=9, color=STEEL)
    ax.grid(axis='x', alpha=0.5)

plt.tight_layout()
plt.savefig('../model/eda_top_genres.png', dpi=150, bbox_inches='tight')
plt.show()

## User Activity Distribution
How many ratings does each user contribute? Typically follows a power-law.

In [ ]:
user_counts = df.groupby(['source', 'user_id']).size().reset_index(name='n_ratings')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (src, color) in zip(axes, SRC_COLORS.items()):
    vals = user_counts[user_counts['source'] == src]['n_ratings']
    ax.hist(vals, bins=50, color=color, edgecolor=CANVAS, linewidth=0.4, alpha=0.9, log=True)
    ax.set_title(f'{src} — Ratings per User  (median={int(vals.median())})')
    ax.set_xlabel('Number of Ratings')
    ax.set_ylabel('User Count (log scale)')
    ax.grid(axis='y', alpha=0.5)

fig.suptitle('User Activity Distribution', fontsize=14, fontweight='semibold', y=1.02)
plt.tight_layout()
plt.savefig('../model/eda_user_activity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nUser activity percentiles (all sources):')
all_user_counts = df.groupby('user_id').size()
print(all_user_counts.quantile([0.25, 0.5, 0.75, 0.90, 0.99]).to_string())

## Item Popularity Distribution
How many ratings does each item receive? Long-tail is expected.

In [ ]:
item_counts = df.groupby(['source', 'item_id']).size().reset_index(name='n_ratings')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (src, color) in zip(axes, SRC_COLORS.items()):
    vals = item_counts[item_counts['source'] == src]['n_ratings']
    ax.hist(vals, bins=50, color=color, edgecolor=CANVAS, linewidth=0.4, alpha=0.9, log=True)
    ax.set_title(f'{src} — Ratings per Item  (median={int(vals.median())})')
    ax.set_xlabel('Number of Ratings')
    ax.set_ylabel('Item Count (log scale)')
    ax.grid(axis='y', alpha=0.5)

fig.suptitle('Item Popularity Distribution (Long Tail)', fontsize=14, fontweight='semibold', y=1.02)
plt.tight_layout()
plt.savefig('../model/eda_item_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

## Mean Rating vs Popularity
Do popular items also get better ratings?

In [ ]:
item_stats = df.groupby(['item_id', 'source']).agg(
    n_ratings  = ('rating', 'count'),
    mean_rating= ('rating', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))

for src, color in SRC_COLORS.items():
    sub = item_stats[item_stats['source'] == src]
    ax.scatter(
        np.log1p(sub['n_ratings']), sub['mean_rating'],
        color=color, alpha=0.25, s=8, label=src,
    )

ax.set_xlabel('log(1 + number of ratings)')
ax.set_ylabel('Mean Rating')
ax.set_title('Item Popularity vs Mean Rating')
ax.legend(frameon=False)
ax.grid(alpha=0.4)

plt.tight_layout()
plt.savefig('../model/eda_rating_vs_popularity.png', dpi=150, bbox_inches='tight')
plt.show()

## Top 10 Most Rated Items

In [ ]:
for src, color in SRC_COLORS.items():
    top = (
        df[df['source'] == src]
        .groupby('title')['rating']
        .agg(n='count', mean='mean')
        .sort_values('n', ascending=False)
        .head(10)
    )
    print(f'\n--- Top 10 Most Rated {src} ---')
    top['mean'] = top['mean'].round(2)
    print(top.to_string())

## Summary

| Observation | Implication |
|---|---|
| Anime ratings skew high (7–8 most common) | Normalization is critical; threshold for 'relevant' should be ≥ 6/10 |
| Movie ratings cluster around 3–4/5 | 3/5 ≈ 'good' — standard Movielens pattern |
| User activity is power-law distributed | Most users rate very few items — cold-start is a real challenge |
| Item popularity is heavily long-tailed | Popularity bias is expected; NCF should beat pure popularity for tail items |
| Comedy & Drama dominate both datasets | Genre filter is meaningful and worth exposing in the UI |